# 01 -- Research Exploration

Cross-Asset Graph Diffusion Signals for Equity Return Prediction.

This notebook is exploratory scaffolding, not the pipeline itself -- the production, tested code
lives in `src/graph_diffusion_signal/` and is driven by `make data` / `make backtest` / `make report`.
Use this notebook to poke at intermediate artefacts (cleaned prices, the correlation graph on a
given date, feature distributions) after running `make data` and `make backtest` at least once.

**Data notice:** if Yahoo Finance was unreachable when `make data` ran, `data/raw/` contains a
calibrated *synthetic* placebo dataset instead of real prices (see `data/raw/SYNTHETIC_DATA_NOTICE.txt`
if present). Everything below still runs the same either way -- just check for that file before
drawing conclusions about real markets from anything you see here.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if (Path.cwd() / "notebooks").exists() is False else Path.cwd()
ROOT = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from graph_diffusion_signal.config import UniverseConfig, BacktestConfig
from graph_diffusion_signal import data as data_mod
from graph_diffusion_signal import graph as graph_mod

uni = UniverseConfig.from_yaml(ROOT / "config" / "universe.yaml")
bt = BacktestConfig.from_yaml(ROOT / "config" / "backtest.yaml")

synthetic_notice = ROOT / "data" / "raw" / "SYNTHETIC_DATA_NOTICE.txt"
print("SYNTHETIC DATA IN USE" if synthetic_notice.exists() else "Real (or previously cached) data in use")

In [ ]:
prices = pd.read_csv(ROOT / "data" / "processed" / "prices_wide.csv", index_col=0, parse_dates=True)
returns = data_mod.compute_simple_returns(prices)
print(prices.shape, prices.index.min(), prices.index.max())
prices.tail()

## Sanity checks on cleaned data

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
(prices / prices.iloc[0]).plot(ax=ax, legend=False, linewidth=0.7, alpha=0.6)
ax.set_title("Normalised price paths, full universe")
ax.set_ylabel("Growth of $1")
plt.show()

print("Daily return summary stats:")
returns.stack().describe()

## A single day's correlation graph

Pick a recent date and look at the rolling-correlation graph the pipeline would have used *as of
that date* (built strictly from returns through the prior trading day -- see `graph.py`).

In [ ]:
universe_tickers = [t for t in uni.all_tickers if t in returns.columns]
returns_universe = returns[universe_tickers]

gparams = graph_mod.GraphParams(
    window=bt["graph"]["correlation_window"],
    top_k=bt["graph"]["top_k"],
    min_abs_corr=bt["graph"]["min_abs_corr"],
)

sample_date = returns_universe.index[-5]  # a recent date with a full trailing window available
loc = returns_universe.index.get_loc(sample_date)
window = returns_universe.iloc[loc - gparams.window : loc]
adj = graph_mod._adjacency_from_window(window, gparams)

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(adj.values, cmap="viridis")
ax.set_xticks(range(len(adj.columns)))
ax.set_xticklabels(adj.columns, rotation=90, fontsize=7)
ax.set_yticks(range(len(adj.index)))
ax.set_yticklabels(adj.index, fontsize=7)
ax.set_title(f"Row-normalised adjacency as of {sample_date.date()}")
plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

## Next steps

Run `make backtest` to produce the full walk-forward results in `reports/results/`, then `make
report` to build `reports/quant_research_report.pdf`. The rest of the statistical analysis (Sharpe,
permutation tests, cost sensitivity, benchmark comparison) lives in `graph_diffusion_signal.metrics`
and is exercised end-to-end by `scripts/run_pipeline.py` -- this notebook intentionally stays light
and exploratory rather than duplicating that logic.